# Utopia API Smoke Test

Notebook minimale per testare la connessione API a Utopia (modelli + chat completion).

## 1) Import e configurazione

In [1]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv


In [3]:
load_dotenv()

UTOPIA_API_KEY = os.getenv("UTOPIA_API_KEY")
CHAT_MODEL = os.getenv("UTOPIA_CHAT_MODEL", "SLURM.gpt-oss:120b")
BASE_URL = "https://utopia.hpc4ai.unito.it"

if not UTOPIA_API_KEY:
    raise RuntimeError("Imposta UTOPIA_API_KEY nel file .env o nelle variabili ambiente")

AUTH_HEADERS = {
    "Authorization": f"Bearer {UTOPIA_API_KEY}"
}

CHAT_HEADERS = {
    **AUTH_HEADERS,
    "Content-Type": "application/json",
}


## 2) Test endpoint modelli (`/ollama/api/tags`)

In [4]:
models_resp = requests.get(
    f"{BASE_URL}/ollama/api/tags",
    headers=AUTH_HEADERS,
    timeout=60,
)
models_resp.raise_for_status()

models_data = models_resp.json()
print("Status code:", models_resp.status_code)
print("Numero modelli trovati:", len(models_data.get("models", [])))
pprint(models_data.get("models", [])[:5])


Status code: 200
Numero modelli trovati: 5
[{'connection_type': 'external',
  'details': {'families': ['gptoss'],
              'family': 'gptoss',
              'format': 'gguf',
              'parameter_size': '20.9B',
              'parent_model': '',
              'quantization_level': 'MXFP4'},
  'digest': '17052f91a42e97930aa6e28a6c6c06a983e6a58dbb00434885a0cf5313e376f7',
  'model': 'SLURM.gpt-oss:20b',
  'modified_at': '2026-01-31T13:35:11+01:00',
  'name': 'gpt-oss:20b',
  'size': 13793441244,
  'urls': [1]},
 {'connection_type': 'external',
  'details': {'families': ['nomic-bert'],
              'family': 'nomic-bert',
              'format': 'gguf',
              'parameter_size': '137M',
              'parent_model': '',
              'quantization_level': 'F16'},
  'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f',
  'model': 'SLURM.nomic-embed-text:latest',
  'modified_at': '2026-01-31T13:35:10+01:00',
  'name': 'nomic-embed-text:latest',
  'size

## 3) Test chat completion (`/api/chat/completions`)

In [5]:
payload = {
    "model": CHAT_MODEL,
    "messages": [
        {"role": "user", "content": "Ciao Utopia, rispondi con: connessione riuscita"}
    ],
}

chat_resp = requests.post(
    f"{BASE_URL}/api/chat/completions",
    headers=CHAT_HEADERS,
    json=payload,
    timeout=180,
)
chat_resp.raise_for_status()

chat_data = chat_resp.json()
print("Status code:", chat_resp.status_code)
print("Risposta modello:")
print(chat_data["choices"][0]["message"]["content"])

chat_data


Status code: 200
Risposta modello:
connessione riuscita


{'id': 'gpt-oss:120b-639649a6-055b-479e-9ab0-9f1b2f5934ee',
 'created': 1771100832,
 'model': 'gpt-oss:120b',
 'choices': [{'index': 0,
   'logprobs': None,
   'finish_reason': 'stop',
   'message': {'role': 'assistant',
    'content': 'connessione riuscita',
    'reasoning_content': 'The user says: "Ciao Utopia, rispondi con: connessione riuscita". They want a response with that phrase. Likely they are testing. The assistant should comply: respond with "connessione riuscita". Probably just that line. No extra.'}}],
 'object': 'chat.completion',
 'usage': {'response_token/s': 169.49,
  'prompt_token/s': 630.74,
  'total_duration': 13300927051,
  'load_duration': 12681108215,
  'prompt_eval_count': 81,
  'prompt_tokens': 81,
  'prompt_eval_duration': 128419825,
  'eval_count': 69,
  'completion_tokens': 69,
  'eval_duration': 407105609,
  'approximate_total': '0h0m13s',
  'total_tokens': 150,
  'completion_tokens_details': {'reasoning_tokens': 0,
   'accepted_prediction_tokens': 0,
   '